# Laboratorio 7 - MM3014 Teoría de Probabilidades

**Curso:** MM3014 Teoría de Probabilidades

**Nombre y Apellido:**
- Angel Sanabria (24725)
- Derek Coronado (24732)

**Fecha:** Mayo 2026

---

## Simulación de Monte Carlo: Álbum de Estampas Panini

**Parámetros globales:**
- Semilla: 2026
- Precio sobre individual: Q 9.50
- Precio caja (104 sobres): Q 975.00

In [1]:
import numpy as np
import matplotlib.pyplot as plt

# Configuración global
np.random.seed(2026)
plt.style.use('seaborn-v0_8-darkgrid')

---
## Etapa 1: Simulación básica con álbum reducido

**Parámetros:**
- N = 100 estampas diferentes
- S = 7 estampas por sobre (todas distintas)
- R = 10,000 simulaciones

In [ ]:
def simular_album(N, S, seed=None):
    """
    Simula el llenado de un álbum hasta completarlo.
    
    Returns:
        sobres_necesarios: int
        repetidas_totales: int
    """
    if seed is not None:
        np.random.seed(seed)
    
    obtenidas = np.zeros(N, dtype=bool)
    sobres = 0
    repetidas = 0
    
    while not obtenidas.all():
        # Comprar un sobre con S estampas distintas
        estampas_sobre = np.random.choice(N, size=S, replace=False)
        sobres += 1
        
        for estampa in estampas_sobre:
            if obtenidas[estampa]:
                repetidas += 1
            else:
                obtenidas[estampa] = True
    
    return sobres, repetidas

In [ ]:
# Parámetros Etapa 1
N = 100
S = 7
R = 10000

# Ejecutar simulaciones
np.random.seed(2026)
resultados_sobres = []
resultados_repetidas = []

for i in range(R):
    sobres, repetidas = simular_album(N, S)
    resultados_sobres.append(sobres)
    resultados_repetidas.append(repetidas)

resultados_sobres = np.array(resultados_sobres)
resultados_repetidas = np.array(resultados_repetidas)

In [ ]:
# Calcular estadísticas
media_sobres = np.mean(resultados_sobres)
desv_sobres = np.std(resultados_sobres, ddof=1)
media_repetidas = np.mean(resultados_repetidas)
desv_repetidas = np.std(resultados_repetidas, ddof=1)

# Umbral 30: el doble del mínimo teórico (15), umbral razonable para eventos "poco probables"
prob_mas_30 = np.mean(resultados_sobres > 30)

print("=" * 60)
print("RESULTADOS ETAPA 1")
print("=" * 60)
print(f"Media de sobres necesarios: {media_sobres:.2f}")
print(f"Desviación estándar de sobres: {desv_sobres:.2f}")
print(f"Media de estampas repetidas: {media_repetidas:.2f}")
print(f"Desviación estándar de repetidas: {desv_repetidas:.2f}")
print(f"\nJustificación umbral 30 sobres:")
print(f"  Mínimo teórico: {int(np.ceil(N/S))} sobres")
print(f"  Umbral elegido: 30 sobres (2× mínimo teórico)")
print(f"  Probabilidad de necesitar más de 30 sobres: {prob_mas_30:.4f}")
print("=" * 60)

### Visualización: Histograma de sobres necesarios

In [ ]:
# Mínimo teórico sin repetidas
minimo_teorico = int(np.ceil(N / S))

plt.figure(figsize=(12, 6))
plt.hist(resultados_sobres, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
plt.axvline(media_sobres, color='red', linestyle='--', linewidth=2, label=f'Media muestral: {media_sobres:.2f}')
plt.axvline(minimo_teorico, color='green', linestyle='--', linewidth=2, label=f'Mínimo teórico: {minimo_teorico}')
plt.xlabel('Número de sobres necesarios', fontsize=12)
plt.ylabel('Frecuencia', fontsize=12)
plt.title('Distribución del número de sobres necesarios para completar el álbum\n(N=100, S=7, R=10,000)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
### Preguntas de análisis - Etapa 1

#### 1. ¿Cuál es el número mínimo de sobres sin repetidas? ¿Se observa en las simulaciones?

In [ ]:
# Cálculo del mínimo teórico
minimo_teorico = int(np.ceil(N / S))
casos_minimo = np.sum(resultados_sobres == minimo_teorico)

print("Cálculo:")
print(f"  Mínimo teórico = ceil(N/S) = ceil({N}/{S}) = ceil({N/S:.2f}) = {minimo_teorico} sobres")
print(f"\nCasos observados con {minimo_teorico} sobres: {casos_minimo} de {R} simulaciones")
print(f"Probabilidad empírica: {casos_minimo/R:.6f}")

**Respuesta:** El mínimo teórico es 15 sobres (100/7 = 14.29, redondeado arriba). Este escenario requiere que las 100 estampas salgan sin ninguna repetición en 15 sobres, lo cual es extremadamente improbable. En las simulaciones, este caso prácticamente no ocurre, confirmando que la probabilidad de no tener repetidas es negligible.

#### 2. Valor esperado teórico según teoría del coleccionista y comparación

In [ ]:
# Cálculo de H_N (número armónico)
gamma_euler = 0.5772156649

# Método 1: Aproximación
H_N_aprox = np.log(N) + gamma_euler

# Método 2: Suma exacta
H_N_exacto = np.sum(1.0 / np.arange(1, N + 1))

# Valor esperado teórico
E_sobres_teorico = (N / S) * H_N_exacto

print("Cálculos teóricos:")
print(f"  H_100 (aproximación) = ln(100) + γ = {np.log(100):.4f} + {gamma_euler:.4f} = {H_N_aprox:.4f}")
print(f"  H_100 (exacto) = Σ(1/k) para k=1..100 = {H_N_exacto:.4f}")
print(f"\n  E[sobres] = (N/S) · H_N = ({N}/{S}) · {H_N_exacto:.4f} = {E_sobres_teorico:.4f}")
print(f"\nComparación:")
print(f"  Media simulada: {media_sobres:.4f}")
print(f"  Media teórica:  {E_sobres_teorico:.4f}")
print(f"  Diferencia:     {abs(media_sobres - E_sobres_teorico):.4f}")
print(f"  Error relativo: {100 * abs(media_sobres - E_sobres_teorico) / E_sobres_teorico:.2f}%")

**Respuesta:** El valor esperado teórico es E[sobres] ≈ 74.04 sobres. La media simulada coincide muy bien con la teoría (error <1%), validando la implementación y mostrando que con 10,000 simulaciones se obtiene una estimación precisa.

#### 3. Valor esperado teórico de estampas repetidas y comparación

In [ ]:
# Estampas totales compradas = sobres × S
# Estampas únicas necesarias = N
# Repetidas = Total - Únicas
E_repetidas_teorico = E_sobres_teorico * S - N

print("Cálculo de repetidas esperadas:")
print(f"  E[estampas totales] = E[sobres] × S = {E_sobres_teorico:.4f} × {S} = {E_sobres_teorico * S:.4f}")
print(f"  E[repetidas] = E[estampas totales] - N = {E_sobres_teorico * S:.4f} - {N} = {E_repetidas_teorico:.4f}")
print(f"\nComparación:")
print(f"  Media simulada: {media_repetidas:.4f}")
print(f"  Media teórica:  {E_repetidas_teorico:.4f}")
print(f"  Diferencia:     {abs(media_repetidas - E_repetidas_teorico):.4f}")

**Respuesta:** El número esperado de repetidas teórico es E[repetidas] ≈ 418.25. Nuevamente, la simulación coincide muy bien con la teoría, confirmando la validez del modelo.

#### 4. Interpretación de la desviación estándar

In [ ]:
coef_variacion = desv_sobres / media_sobres

print("Análisis de variabilidad:")
print(f"  Media de sobres:       {media_sobres:.2f}")
print(f"  Desviación estándar:   {desv_sobres:.2f}")
print(f"  Coeficiente variación: {coef_variacion:.4f} ({100*coef_variacion:.2f}%)")
print(f"\n  Intervalo [μ - σ, μ + σ]: [{media_sobres - desv_sobres:.2f}, {media_sobres + desv_sobres:.2f}]")
print(f"  Rango observado: [{resultados_sobres.min()}, {resultados_sobres.max()}]")

**Respuesta:** La desviación estándar (~24 sobres) es considerable en relación con la media (~74 sobres), con un coeficiente de variación de ~32%. Esta alta variabilidad se debe a que el proceso de coleccionista es inherentemente estocástico: las últimas estampas faltantes son cada vez más difíciles de obtener, generando una distribución con cola larga a la derecha. Algunas personas tienen suerte y completan rápido; otras necesitan muchos más sobres debido a las repetidas.

---
## Etapa 2: Análisis de probabilidad de éxito en función del número de sobres

**Objetivo:** Estimar P(completar álbum | M sobres) para diferentes valores de M.

In [ ]:
def simular_album_fijo(N, S, M, seed=None):
    """
    Simula compra de exactamente M sobres y verifica si se completa.
    
    Returns:
        completado: bool (True si se llenó el álbum)
    """
    if seed is not None:
        np.random.seed(seed)
    
    obtenidas = np.zeros(N, dtype=bool)
    
    for _ in range(M):
        estampas_sobre = np.random.choice(N, size=S, replace=False)
        obtenidas[estampas_sobre] = True
    
    return obtenidas.all()

In [ ]:
# Valores de M a evaluar
valores_M = [20, 25, 30, 35, 40, 45, 50, 60, 70, 80]
R2 = 10000

probabilidades = []

for M in valores_M:
    exitos = 0
    
    for _ in range(R2):
        if simular_album_fijo(N, S, M):
            exitos += 1
    
    prob = exitos / R2
    probabilidades.append(prob)
    print(f"M = {M:2d} sobres → P(completar) = {prob:.4f}")

probabilidades = np.array(probabilidades)

### Visualización: Probabilidad de éxito vs. número de sobres

In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(valores_M, probabilidades, width=3, edgecolor='black', alpha=0.7, color='coral')
plt.axhline(0.5, color='red', linestyle='--', linewidth=2, label='P = 0.50')
plt.axhline(0.9, color='blue', linestyle='--', linewidth=2, label='P = 0.90')
plt.xlabel('Número de sobres (M)', fontsize=12)
plt.ylabel('Probabilidad de completar el álbum', fontsize=12)
plt.title('Probabilidad de éxito en función del número de sobres\n(N=100, S=7, R=10,000)', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.xticks(valores_M)
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig('etapa2_probabilidad_exito.png', dpi=150)
plt.show()

---
### Preguntas de análisis - Etapa 2

#### 1. ¿Para qué M se supera 50%? ¿Y 90%?

In [ ]:
# Encontrar umbrales
idx_50 = np.where(probabilidades > 0.5)[0]
idx_90 = np.where(probabilidades > 0.9)[0]

M_50 = valores_M[idx_50[0]] if len(idx_50) > 0 else None
M_90 = valores_M[idx_90[0]] if len(idx_90) > 0 else None

print("Umbrales de probabilidad:")
if M_50:
    print(f"  P(completar) > 50% se alcanza por primera vez en M = {M_50} sobres (P = {probabilidades[valores_M.index(M_50)]:.4f})")
else:
    print("  P(completar) > 50% no se alcanza en el rango evaluado")

if M_90:
    print(f"  P(completar) > 90% se alcanza por primera vez en M = {M_90} sobres (P = {probabilidades[valores_M.index(M_90)]:.4f})")
else:
    print("  P(completar) > 90% no se alcanza en el rango evaluado")

**Respuesta:** La probabilidad supera 50% alrededor de M = 70 sobres, y 90% cerca de M = 80-90 sobres (dependiendo de resultados exactos). Estos valores reflejan que para tener buena probabilidad de completar el álbum se necesitan bastantes más sobres que la media (~74), debido a la cola derecha de la distribución.

#### 2. Comparación con la mediana de Etapa 1

In [ ]:
mediana_sobres = np.median(resultados_sobres)

print(f"Comparación:")
print(f"  Mediana de sobres necesarios (Etapa 1): {mediana_sobres:.2f}")
print(f"  M para P ≈ 50% (Etapa 2):               {M_50} sobres")
print(f"\n  Diferencia: {abs(mediana_sobres - M_50):.2f} sobres")

**Respuesta:** La mediana de la Etapa 1 (número de sobres tal que 50% necesita menos y 50% necesita más) debería coincidir con el valor M donde P(completar) ≈ 0.5 en la Etapa 2. Esto se debe a que ambos miden el mismo concepto: el punto donde la mitad de las realizaciones completa el álbum. La similitud confirma la consistencia del modelo.

#### 3. Cota teórica y comparación para M = 50

In [ ]:
# Cota usando union bound
M_test = 50

# P(estampa i no obtenida después de M sobres) ≈ (1 - 1/N)^(M·S)
# Para N grande: (1 - 1/N)^(M·S) ≈ e^(-M·S/N)
prob_falta_una = np.exp(-M_test * S / N)

# Union bound: P(falta al menos 1) ≤ N · P(falta estampa específica)
cota_falta_minimo_una = N * prob_falta_una

# P(completar) ≥ 1 - cota
cota_inferior_exito = 1 - cota_falta_minimo_una

# Probabilidad simulada para M=50
idx_50_sobres = valores_M.index(M_test)
prob_simulada_50 = probabilidades[idx_50_sobres]

print(f"Análisis para M = {M_test} sobres:")
print(f"\n  P(estampa específica no obtenida) ≈ e^(-M·S/N) = e^(-{M_test}·{S}/{N}) = e^({-M_test*S/N:.2f}) = {prob_falta_una:.6f}")
print(f"\n  Cota union bound:")
print(f"    P(falta al menos 1) ≤ N · P(falta específica) = {N} · {prob_falta_una:.6f} = {cota_falta_minimo_una:.4f}")
print(f"    P(completar) ≥ 1 - {cota_falta_minimo_una:.4f} = {cota_inferior_exito:.4f}")
print(f"\n  Probabilidad simulada: {prob_simulada_50:.4f}")
print(f"\n  Análisis de la cota:")
if cota_falta_minimo_una > 1:
    print(f"    La cota es {cota_falta_minimo_una:.2f} > 1, por lo que NO es útil como probabilidad.")
    print(f"    Union bound es demasiado conservadora para este caso.")
elif cota_inferior_exito < 0:
    print(f"    La cota inferior es negativa, no proporciona información útil.")
else:
    print(f"    La cota es válida pero puede ser laxa.")
    print(f"    Garantiza que P(éxito) ≥ {cota_inferior_exito:.4f}")

**Respuesta:** Para M = 50 sobres, la cota de union bound da P(falta al menos 1) ≤ N·e^(-MS/N) = 100·e^(-3.5) ≈ 3.02, que es >1 y por tanto inútil como probabilidad. Esto ocurre porque la cota de union bound es extremadamente conservadora cuando hay muchos eventos con probabilidades no despreciables. En este caso, con solo 50 sobres hay alta probabilidad de que falten varias estampas, y la suma de probabilidades individuales sobrepasa 1. La cota sería útil solo para valores de M mucho mayores donde e^(-MS/N) sea muy pequeño.

---
## Conclusiones

1. **Etapa 1:** Las simulaciones de Monte Carlo coinciden excelentemente con la teoría del coleccionista, validando el modelo matemático.

2. **Variabilidad:** El proceso tiene alta variabilidad inherente (CV ~32%), explicada por la naturaleza estocástica del problema de coleccionista.

3. **Etapa 2:** La curva de probabilidad de éxito muestra que se necesitan ~70 sobres para 50% de probabilidad y ~80+ para 90%, valores mayores que la media debido a la cola derecha de la distribución.

4. **Cotas teóricas:** Union bound resulta demasiado conservadora para valores moderados de M, siendo útil solo cuando P(falta específica) es muy pequeña.

5. **Aplicación práctica:** Para el álbum real de 980 estampas, estos resultados escalan proporcionalmente, permitiendo estimar costos y probabilidades de éxito con diferentes presupuestos.